In [ ]:
import cv2
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import os
from ultralytics import YOLO
import numpy as np

### Cargar imagen y Ground Truth

In [ ]:
# Rutas
ANN_FILE = 'datasets/coco/annotations/instances_val2017.json'
IMG_DIR = 'datasets/coco/val2017'
TARGET_ID = 139

# Cargar anotaciones COCO
coco = COCO(ANN_FILE)
img_info = coco.loadImgs(TARGET_ID)[0]
img_path = os.path.join(IMG_DIR, img_info['file_name'])

# Cargar imagen con OpenCV
img = cv2.imread(img_path)
img_gt = img.copy()

# Dibujar Ground Truth (Cajas Verdes)
ann_ids = coco.getAnnIds(imgIds=img_info['id'])
anns = coco.loadAnns(ann_ids)

for ann in anns:
    x, y, w, h = [int(v) for v in ann['bbox']]
    cat_id = ann['category_id']
    cat_name = coco.loadCats(cat_id)[0]['name']
    cv2.rectangle(img_gt, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(img_gt, f"GT: {cat_name}", (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

print(f"Imagen cargada: {img_path}")

### Ejecutar predicciones con todos los modelos

In [ ]:
models_dict = {
    'n': 'yolov8n.pt',
    's': 'yolov8s.pt',
    'm': 'yolov8m.pt',
    'l': 'yolov8l.pt',
    'x': 'yolov8x.pt'
}

best_model_name = None
max_detections = -1
best_result = None

print("Ejecutando inferencia...")
for key, model_path in models_dict.items():
    model = YOLO(model_path)
    # Ejecutar inferencia
    results = model(img_path, verbose=False)

    num_detections = len(results[0].boxes)
    print(f"Modelo YOLOv8{key}: {num_detections} objetos detectados")

    if num_detections > max_detections:
        max_detections = num_detections
        best_model_name = key
        best_result = results[0]

### Dibujar resultado del mejor modelo y comparar

In [ ]:
# Dibujar Predicciones (Cajas Rojas)
img_pred = img.copy()
print(f"\nMejor modelo seleccionado: YOLOv8{best_model_name}")

for box in best_result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    class_name = best_result.names[cls_id]

    cv2.rectangle(img_pred, (x1, y1), (x2, y2), (0, 0, 255), 2)
    label = f"{class_name} {conf:.2f}"
    cv2.putText(img_pred, label, (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

# Visualización lado a lado
plt.figure(figsize=(18, 10))

plt.subplot(1, 2, 1)
plt.title("Ground Truth (Verde)")
plt.imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Predicción YOLOv8{best_model_name} (Rojo)")
plt.imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.tight_layout()
plt.savefig("comparacion_139.png")
plt.show()